# Secure Aggregation: Privacy Without a Trusted Server

FL alone does not prevent the server from inspecting individual client updates. Secure aggregation uses cryptographic protocols so the server learns only the aggregate — not any individual client's contribution.

## The Secure Aggregation Problem

In standard FL, the server sees each client's gradient update in plaintext. A curious server can:
- Infer sensitive attributes from gradient updates
- Perform gradient inversion attacks to reconstruct training data
- Track individual client behavior over rounds

Secure aggregation (Bonawitz et al. 2017) uses secret sharing and masking so that only the sum of client updates is revealed to the server — no individual update is visible.

In [ ]:
import random, math
from dataclasses import dataclass, field

@dataclass
class SecAggClient:
    client_id: int
    gradient: list
    pairwise_masks: dict = field(default_factory=dict)

def generate_pairwise_masks(client_ids: list, n_params: int, seed: int = 42) -> dict:
    rng = random.Random(seed)
    masks = {}
    for i, ci in enumerate(client_ids):
        for j, cj in enumerate(client_ids):
            if i < j:
                # Each pair shares a mask; one adds, the other subtracts
                mask = [rng.gauss(0, 1) for _ in range(n_params)]
                masks[(ci, cj)] = mask
    return masks

def mask_gradient(client: SecAggClient, all_masks: dict) -> list:
    masked = list(client.gradient)
    for (ci, cj), mask in all_masks.items():
        if ci == client.client_id:
            masked = [masked[i] + mask[i] for i in range(len(masked))]
        elif cj == client.client_id:
            masked = [masked[i] - mask[i] for i in range(len(masked))]
    return masked

def secure_aggregate(clients: list, all_masks: dict) -> list:
    masked_updates = [mask_gradient(c, all_masks) for c in clients]
    n_params = len(clients[0].gradient)
    # Sum of masked gradients = sum of unmasked gradients (masks cancel out)
    aggregated = [sum(u[i] for u in masked_updates)/len(clients) for i in range(n_params)]
    return aggregated

def verify_secure_aggregation(clients: list, all_masks: dict) -> dict:
    true_aggregate = [sum(c.gradient[i] for c in clients)/len(clients)
                       for i in range(len(clients[0].gradient))]
    secure_agg = secure_aggregate(clients, all_masks)
    error = math.sqrt(sum((a-b)**2 for a,b in zip(true_aggregate, secure_agg)))
    return {'true_aggregate': true_aggregate, 'secure_aggregate': secure_agg,
            'error': error, 'masks_cancel': error < 1e-8}

rng = random.Random(42)
client_ids = list(range(5))
n_params = 3
true_grads = [[rng.gauss(1.0, 0.1) for _ in range(n_params)] for _ in range(5)]
clients = [SecAggClient(i, true_grads[i]) for i in range(5)]
all_masks = generate_pairwise_masks(client_ids, n_params)

result = verify_secure_aggregation(clients, all_masks)
print('Secure Aggregation Verification:')
print(f'  True aggregate:   {[round(x,4) for x in result["true_aggregate"]]}')
print(f'  Secure aggregate: {[round(x,4) for x in result["secure_aggregate"]]}')
print(f'  Numerical error:  {result["error"]:.2e}')
print(f'  Masks cancel out: {result["masks_cancel"]}')
print('\nServer sees masked updates (individual updates hidden):')
for c in clients:
    masked = mask_gradient(c, all_masks)
    print(f'  Client {c.client_id}: {[round(x,3) for x in masked]} (true: {[round(x,3) for x in c.gradient]})')

## MPC-Lite and Production Secure Aggregation

The masking approach above is a simplified version. Production secure aggregation (as in TensorFlow Federated and PySyft) uses:

**Secret sharing**: each client's update is split into shares distributed among other clients, so no single party (including the server) has the full update.

**Dropout handling**: real FL has client dropout. The protocol must handle clients that drop out mid-round without revealing partial information.

**Authenticated masking**: pairwise masks must be authenticated (each pair verifies the other's mask) to prevent a malicious client from injecting false masks.

Secure aggregation increases communication overhead by a factor of the number of clients (each pair exchanges masks). For large-scale deployments (thousands of clients), this is significant — practical systems use hierarchical aggregation to limit the overhead.

## Series 16 Recap: Privacy-Preserving ML

Across these 9 notebooks:
1. **Privacy threat model**: MIA, attribute inference, model inversion, data extraction
2. **Differential privacy**: ε-δ DP, Laplace/Gaussian mechanisms, composition
3. **DP-SGD**: gradient clipping, noise injection, privacy-utility tradeoff
4. **Privacy auditing**: LiRA attack, empirical ε estimation
5. **Federated learning**: FedAvg, non-IID challenges
6. **Federated optimization**: FedProx, SCAFFOLD — fixing client drift
7. **Byzantine robustness**: Krum, Trimmed Mean, FLTrust
8. **Flower framework**: production FL simulation
9. **Secure aggregation**: masking protocols, MPC-lite

The key insight: privacy in ML is not a binary property — it is a spectrum with quantifiable tradeoffs. DP gives you a number (ε). FL reduces data centralization. Secure aggregation protects individual updates. Combining all three gives defense in depth.